In [28]:
from numpy.ma.extras import average
from sympy.physics.units import farad


######################
## 拆解 - 重构 - 扩展 ##
## 拆解 - 重构 - 扩展 ##
## 拆解 - 重构 - 扩展 ##
######################

#1-基础类：定义一个学生类
#属性：姓名、年龄、成绩
#方法：介绍自己、获取成绩
class Student:
    def __init__(self,name,age,score):
        self.name = name
        self.age = age
        self.score = score
    def introduce(self):
        return f"我是{self.name},今年{self.age}岁,这次考试成绩为{self.score}分"
    def get_score(self):
        return self.score
s = Student("小明",20,90)
print(s.introduce())
print(s.get_score())

我是小明,今年20岁,这次考试成绩为90分
90


In [64]:
from datetime import datetime
import json
class Score:
    """成绩类-拆解：独立管理成绩"""
    def __init__(self,subject,score):
        self.subject = subject
        self.score = self._validate_score(score)
        self.update_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S") #时间戳
    def _validate_score(self,score):
        """验证成绩有效性"""
        if not 0 <= score <=100:
            raise ValueError(f"成绩{score}不在有效范围（0～100）")
        return score
    def update_score(self,new_score):
        """更新成绩"""
        self.score = self._validate_score(new_score)
        self.update_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        return f"{self.subject}成绩已经更新为{self.score}分"
    def get_grade(self):
        """获取等级"""
        if self.score >=90:
            return "A"
        elif self.score >=80:
            return "B"
        elif self.score >=70:
            return "C"
        elif self.score >=60:
            return "D"
        else:
            return "F"
    def to_dict(self):
        return{
            "subject":self.subject,
            "score":self.score,
            "grade":self.get_grade(),
            "update_time":self.update_time
        }
class Student:
    """学生类 - 重构：优化结构和职责"""
    def __init__(self,name,age,sex,scores=None):
        self.name = name
        self.age = self._validate_age(age)
        self.sex = self._validate_sex(sex)
        self.scores ={}    #多课目成绩管理
        if scores:
            for subject,score in scores.items():
                self.add_score(subject,score)
        self.create_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    def _validate_age(self,age):
        """验证年龄有效性"""
        if not 5 <= age <= 28:
            raise ValueError(f"年龄{age}不在有效范围（5～28岁）之间")
        return age
    def _validate_sex(self,sex):
        """验证性别有效性"""
        if sex not in ["男","女"]:
            raise ValueError(f"性别{sex}必须是男或女")
        return sex
    def add_score(self,subject,score):
        """添加科目成绩"""
        self.scores[subject] = Score(subject,score)
        return f"已添加{subject}成绩:{score}分"
    def update_score(self,subject,new_score):
        """更新指定科目成绩"""
        if subject not in self.scores:
            raise KeyError(f"科目{subject}不存在")
        return self.scores[subject].update_score(new_score)
    def get_average_score(self):
        """计算平均分"""
        if not self.scores:
            return 0
        # total = sum(s.score for s in self.scores.values())
        total = 0
        for subject_name,score_object in self.scores.items():
            total += score_object.score
        return round(total / len(self.scores),2)
    def get_best_subject(self):
        """获取最高分科目"""
        if not self.scores:
            return None
        best = max(self.scores.items(),key=lambda x:x[1].score)
        return best[0],best[1].score
    def introduce(self):
        """自我介绍"""
        avg_score = self.get_average_score()
        intro = f"大家好，我叫{self.name},今年{self.age}岁了，{self.sex}生"
        if self.scores:
            intro += f"\n共有{len(self.scores)}门科目成绩"
            intro += f"\n平均成绩:{avg_score}分"
            best_subject,best_score = self.get_best_subject()
            intro += f"\n最高分:{best_subject},{best_score}分"
        return intro
    def get_all_scores(self):
        """获取成绩详情"""
        if not self.scores:
            return "暂无成绩记录"
        result = f"\n{'='*40}\n{self.name}的成绩单\n{'='*40}"
        for subject,score_obj in self.scores.items():
            result += f"\n{subject}:{score_obj.score}分(等级：{score_obj.get_grade()})"
        result += f"\n{'='*40}"
        result += f"\n平均分:{self.get_average_score()}分"
        return result
    def to_dict(self):
        """专为字典(用于序列化)"""
        return{
            "name":self.name,
            "age":self.age,
            "sex":self.sex,
            "scores":{subj:sc.to_dict() for subj,sc in self.scores.items()},
            "average_score":self.get_average_score(),
            "create_time":self.create_time
        }
    def save_to_file(self,filename=None):
        """保存到文件"""
        if not filename:
            filename = f"{self.name}_data.json"
        with open(filename,'w',encoding="utf-8") as f:
            json.dump(self.to_dict(),f,ensure_ascii=False,indent=2)
        return f"数据已保存到{filename}"
    @classmethod
    def from_dict(cls,data):
        """"从字典创建学生对象"""
        student = cls(data['name'],data['age'],data['sex'])
        for subject,score_data in data.get('scores',{}).items():
            student.scores[subject] = Score(subject,score_data['score'])
        return student

class ClassManager:
    """班级管理类 - 扩展：管理多个学生"""
    def __init__(self,class_name):
        self.class_name = class_name
        self.students = {}
    def add_student(self,student):
        """添加学生"""
        self.students[student.name] = student
        return f"学生{student.name}已经加入{self.class_name}"
    def remove_student(self,name):
        """移除学生"""
        if name in self.students:
            del self.students[name]
            return f"学生{name}已移除"
        return f"未找到学生{name}"
    def get_student(self,name):
        """查询学生"""
        return self.students.get(name,None)
    def get_class_average(self):
        """计算班级平均分"""
        if not self.students:
            return 0
        total_avg = sum(s.get_average_score() for s in self.students.values())
        # total_avg = 0
        # for s_name,s_object in self.students.items():
        #     total_avg += s_object.get_average_score()

        return round(total_avg/len(self.students),2)
    def get_ranking(self):
        """获取班级排名"""
        ranking = sorted(
            self.students.items(),
            key = lambda x:x[1].get_average_score(),
            reverse = True
        )
        result = f"\n{'='*50}\n{self.class_name}成绩排名\n{'='*50}"
        for rank,(name,student) in enumerate(ranking,1):
            result += f"\n第{rank}名：{name} - 平均分{student.get_average_score()}"
        result +=f"\n{'='*50}"
        return result
    def get_statistics(self):
        """班级统计信息"""
        if not self.students:
            return "班级暂无学生"
        averages = [s.get_average_score() for s in self.students.values()]
        stats = f"\n📊{self.class_name}统计信息"
        stats += f"\n学生总数：{len(self.students)}"
        stats += f"\n班级平均分：{self.get_class_average()}"
        stats += f"\n最高平均分:{max(averages)}"
        stats += f"\n最低平均分:{min(averages)}"
        return stats
    def save_class_data(self,filename=None):
        """保存班级数据"""
        if not filename:
            filename = f"{self.class_name}_data.json"
        data = {
            "class_name":self.class_name,
            "students":{name:stu.to_dict() for name,stu in self.students.items()}
        }
        with open(filename,'w',encoding="utf-8") as f:
            json.dump(data,f,ensure_ascii=False,indent=2)
        return f"班级数据已经保存到{filename}"

#=======================使用示例=======================

if __name__ == "__main__":
    print("="*60)
    print("🎓学生管理系统")
    print("="*60)

    #1.创建单个学生（向后兼容原功能）
    print("\n【1】创建学生对象")
    student1 = Student('姚闯',18,'男',{'数学':100})
    print(student1.introduce())
    print(student1.get_all_scores())

    #2.展示重构后的强大功能
    print("\n【2】多科目管理")
    student2 = Student('李华',17,'女',{
        '语文':85,
        '数学':92,
        '英语':78
    })
    print(student2.introduce())
    print(student2.get_all_scores())

    #3.更新成绩
    print("\n【3】更新成绩")
    print(student2.update_score('英语',88))
    print(student2.get_all_scores())

    #4.班级管理
    print("\n【4】班级管理功能")
    class_manager = ClassManager('高三（1）班')
    class_manager.add_student(student1)
    class_manager.add_student(student2)

    print(class_manager.get_ranking())
    print(class_manager.get_statistics())

    #5.数据持久化
    print("\n【5】数据保存")
    print(student1.save_to_file())
    print(class_manager.save_class_data())

    print("\n✅ 拆解-重构-扩展完成")
    print(" *拆解：Score类独立管理成绩")
    print(" *重构：Student类优化结构和验证")
    print(" *扩展：ClassManager管理整个班级")


🎓学生管理系统

【1】创建学生对象
大家好，我叫姚闯,今年18岁了，男生
共有1门科目成绩
平均成绩:100.0分
最高分:数学,100分

姚闯的成绩单
数学:100分(等级：A)
平均分:100.0分

【2】多科目管理
大家好，我叫李华,今年17岁了，女生
共有3门科目成绩
平均成绩:85.0分
最高分:数学,92分

李华的成绩单
语文:85分(等级：B)
数学:92分(等级：A)
英语:78分(等级：C)
平均分:85.0分

【3】更新成绩
英语成绩已经更新为88分

李华的成绩单
语文:85分(等级：B)
数学:92分(等级：A)
英语:88分(等级：B)
平均分:88.33分

【4】班级管理功能

高三（1）班成绩排名
第1名：姚闯 - 平均分100.0
第2名：李华 - 平均分88.33

📊高三（1）班统计信息
学生总数：2
班级平均分：94.16
最高平均分:100.0
最低平均分:88.33

【5】数据保存
数据已保存到姚闯_data.json
班级数据已经保存到高三（1）班_data.json

✅ 拆解-重构-扩展完成
 *拆解：Score类独立管理成绩
 *重构：Student类优化结构和验证
 *扩展：ClassManager管理整个班级


In [29]:
#2.私有属性与封装
#成绩设为私有
#提供set_score做合法性判断
class Student:
    def __init__(self,name):
        self.name = name
        self._score = 0
    def set_score(self,score):
        if 0 <= score <= 100:
            self._score = score
        else:
            raise ValueError("成绩必须0-100")
    def get_score(self):
        return self._score
s = Student("小红")
s.set_score(88)
print(s.get_score())

88


In [30]:
#3.类方法、静态方法
class Math:
    PI = 3.14159
    @classmethod
    def circle_area(cls,r):
        return cls.PI*r*r
    @staticmethod
    def add(a,b):
        return a+b
print(Math.circle_area(2))
print(Math.add(3,5))

12.56636
8


In [31]:
#4.继承：员工->程序员
class Employee:
    def __init__(self,name,salary,job):
        self.name = name
        self.salary = salary
        self.job = job
    def work(self):
        return f"{self.name}认真工作中"
class Programmer(Employee):
    def __init__(self,name,salary,job,language):
        super().__init__(name,salary,job)
        self.language = language
    def code(self):
        return f"{self.name}的工作是{self.job},每个月的薪资是{self.salary}元,擅长用{self.language}写代码"
p = Programmer("姚闯",20000,"AI算法工程师","Python")
print(p.work())
print(p.code())

姚闯认真工作中
姚闯的工作是AI算法工程师,每个月的薪资是20000元,擅长用Python写代码


In [32]:
#5.多态：不同动物的叫声
class Animal:
    def speak(self): #实例方法
        pass
class Dog(Animal):
    def speak(self):
        return "汪汪汪"
class Cat(Animal):
    def speak(self):
        return "喵喵喵"
class Bird(Animal):
    def speak(self):
        return "叽叽喳喳"
def make_speak(animal:Animal):  #多态函数
    print(animal.speak())
make_speak(Dog())
make_speak(Cat())
make_speak(Bird())

汪汪汪
喵喵喵
叽叽喳喳


In [33]:
#6.组合（has-a）:电脑有CPU
class CPU:
    def __init__(self,brand="Intel"):
        self.brand = brand
    def info(self):
        return f"CPU:{self.brand}"
class Computer:
    def __init__(self):
        self.cpu = CPU()
    def show(self):
        print(self.cpu.info())
c = Computer()
c.show()

CPU:Intel


In [34]:
#7.魔术方法：str len eq
class Book:
    def __init__(self,name,pages):
        self.name = name
        self.pages = pages
    def __str__(self):
        return f"《{self.name}》"
    def __len__(self):
        return self.pages
    def __eq__(self,other):
        return self.pages == other.pages
b1 = Book("Python入门",300)
b2 = Book("进阶",300)
print(b1)
print(len(b1))
print(b1 == b2)


《Python入门》
300
True


In [35]:
#8.实战小项目：简单银行账户类
class Account:
   def __init__(self,user_id,balance=0):
       self.id = user_id
       self.balance = balance
   def deposit(self,money):    #存钱
       if money > 0:
           self.balance += money
   def withdraw(self,money):   #取钱
       if 0 < money <= self.balance:
           self.balance -= money
   def __str__(self):
       return f"账户{self.id},余额:{self.balance}"
acc = Account("1001")
acc.deposit(1000)
acc.withdraw(300)
print(acc)
acc1 = Account("1002")
acc1.deposit(10000)
acc1.withdraw(5000)
print(acc1)

账户1001,余额:700
账户1002,余额:5000
